# Planning Failure Modes and Mitigation

Planning agents decompose tasks before executing.
But the planning layer itself introduces three distinct failure modes that
don't exist in flat agent loops.

This demonstration shows each failure mode concretely, then shows
a targeted mitigation. Same use case throughout: DocuFlow AI German market entry.


**Failure modes:**

| Failure Mode | Symptom | Root Cause |
|---|---|---|
| **Rigid planning** | New evidence ignored; stale plan executed | No replanning triggers |
| **Over-decomposition** | 12 micro-subtasks; context window exhausted | No subtask budget |
| **No step budget** | Single subtask calls 9 tools; others starved | No per-subtask limit |

## Setup

In [ ]:
import os, json, time
from google import genai
from google.genai import types

# os.environ["GEMINI_API_KEY"] = ""

client = genai.Client()
MODEL = "gemini-2.5-flash-lite"

## Shared Tools and Helpers

We will use the DocuFlow AI tool set again. A lightweight executor helper handles single-subtask execution.

In [50]:
# Tool implementations

def search_regulations(country: str, topic: str) -> str:
    db = {
        ("germany", "data protection"): (
            "GDPR + BDSG apply. Data minimisation (Art.5), privacy-by-design (Art.25). "
            "Employee-data rules stricter under BDSG. "
            "Regulated-sector clients require DE data residency. "
            "Compliance setup: 6-9 months, EUR 120-180K."
        ),
        ("germany", "cloud security"): (
            "BSI C5 required for enterprise/public-sector sales. "
            "Audit timeline: 9-15 months. Cost: EUR 80-140K audit + EUR 60-90K engineering. "
            "ISO 27001 is prerequisite."
        ),
        ("germany", "electronic signatures"): (
            "eIDAS governs e-signatures. QES = legal equivalent of handwritten signature. "
            "Integration with German TSP (D-Trust/DocuSign) needed. "
            "Effort: 2-3 months, EUR 15-25K."
        ),
        ("germany", "e-invoicing"): (
            "E-Rechnungspflicht in force Jan 2025 for B2B >EUR 250K. "
            "Requires EN 16931-compliant XML invoice support. "
            "Implementation: 3-5 months, EUR 40-60K."
        ),
    }
    for (c, t), v in db.items():
        if c in country.lower() and t in topic.lower():
            return v
    return f"No data: country='{country}', topic='{topic}'"

def get_market_data(country: str, sector: str) -> str:
    if "germany" in country.lower() and any(k in sector.lower() for k in ["document","ecm","content"]):
        return (
            "Germany ECM 2024: EUR 1.42B, CAGR 14.2%. "
            "Cloud adoption 41% (vs 67% UK). Addressable segment EUR 380M. "
            "Top verticals: manufacturing 28%, financial services 22%."
        )
    return f"No market data for country='{country}', sector='{sector}'"

def get_competitor_info(product_type: str, region: str) -> str:
    if "german" in region.lower() and any(k in product_type.lower() for k in ["document","ecm","content"]):
        return (
            "German ECM market: DocuWare 23%, ELO 18%, d.velop 11%, M-Files 8%. "
            "DocuWare weakness: legacy architecture. "
            "Opportunity: AI-native differentiation vs DocuWare/ELO."
        )
    return f"No competitor data for product='{product_type}', region='{region}'"

def get_technical_requirements(feature: str, region: str) -> str:
    if "data residency" in feature.lower() and "german" in region.lower():
        return (
            "Minimum: EU-region storage (Frankfurt eu-central-1). "
            "Enterprise tier: single-tenant DE deployment often required. "
            "Cost delta: +EUR 180-240K/year. Timeline: 4-6 months."
        )
    if any(k in feature.lower() for k in ["localisation","localization","german language"]):
        return (
            "Full UI translation: ~12,000 strings. "
            "Legal templates require DE-qualified reviewer (not literal translation). "
            "Effort: 3-4 months, EUR 45-65K."
        )
    return f"No technical data for feature='{feature}', region='{region}'"

def estimate_financial_impact(metric: str, scenario: str) -> str:
    if "investment" in metric.lower() or "cost" in metric.lower():
        return (
            "Year 1 investment: EUR 805K-1.1M (base). "
            "Breakdown: compliance EUR 200-320K, infra EUR 180-240K, "
            "localisation EUR 45-65K, GTM EUR 380-480K."
        )
    if "revenue" in metric.lower() or "arr" in metric.lower():
        return (
            "Year 1 ARR: EUR 475K-1.14M depending on scenario. "
            "Year 3 base case: EUR 6.3M ARR. Payback: 28-34 months."
        )
    return f"No financial data for metric='{metric}', scenario='{scenario}'"

def search_news(topic: str, focus: str) -> str:
    if "docuware" in topic.lower() or ("competitor" in topic.lower() and "german" in focus.lower()):
        return (
            "Q1 2025: DocuWare announced DocuWare AI (GPT-4 document classification, H2 2025 GA). "
            "Jan 2025: ELO raised EUR 45M Series B. "
            "Feb 2025: d.velop acquired Konfuzio AI for EUR 12M. "
            "Window for AI first-mover advantage narrows — target H2 2025 soft launch."
        )
    if "regulation" in topic.lower() or "gdpr" in topic.lower():
        return (
            "Jan 2025: E-Rechnungspflicht in force for B2B >EUR 250K. "
            "Nov 2024: EU AI Act transparency obligations for document AI tools. "
            "Dec 2024: BfDI issued EUR 1.2M fine for inadequate DPA agreements."
        )
    return f"No news for topic='{topic}', focus='{focus}'"

TOOLS = {
    "search_regulations":         search_regulations,
    "get_market_data":            get_market_data,
    "get_competitor_info":        get_competitor_info,
    "get_technical_requirements": get_technical_requirements,
    "estimate_financial_impact":  estimate_financial_impact,
    "search_news":                search_news,
}

In [51]:
# Tool spec builder 
def build_tool_spec(names: list) -> list:
    defs = {
        "search_regulations":         ("Return regulatory requirements for a country and topic.",
                                       {"country": "str", "topic": "str"}),
        "get_market_data":            ("Return market size and growth data.",
                                       {"country": "str", "sector": "str"}),
        "get_competitor_info":        ("Return competitive landscape.",
                                       {"product_type": "str", "region": "str"}),
        "get_technical_requirements": ("Return technical/infra requirements.",
                                       {"feature": "str", "region": "str"}),
        "estimate_financial_impact":  ("Return financial estimates.",
                                       {"metric": "str", "scenario": "str"}),
        "search_news":                ("Return recent news for a topic.",
                                       {"topic": "str", "focus": "str"}),
    }
    tools = []
    for name in names:
        if name not in defs:
            continue
        desc, params = defs[name]
        props = {k: types.Schema(type=types.Type.STRING) for k in params}
        tools.append(types.Tool(function_declarations=[
            types.FunctionDeclaration(
                name=name,
                description=desc,
                parameters=types.Schema(
                    type=types.Type.OBJECT,
                    properties=props,
                    required=list(params.keys())
                )
            )
        ]))
    return tools

In [52]:
# Shared decomposer 
def decompose(task: str, instruction: str = None, max_subtasks: int = None) -> list:
    """Call the LLM decomposer. Returns list of subtask dicts."""
    base = (
        'You are a planning agent. Given a task, output ONLY valid JSON — no markdown fences.\n'
        'Format: {"plan": [{"id": "s1", "goal": "...", "tools": [...], "priority": 1}]}\n'
        'Available tools: search_regulations, get_market_data, get_competitor_info,\n'
        '  get_technical_requirements, estimate_financial_impact, search_news\n'
    )
    if max_subtasks:
        base += f'IMPORTANT: Produce EXACTLY {max_subtasks} subtasks — no more.\n'
    if instruction:
        base += instruction

    response = client.models.generate_content(
        model=MODEL,
        contents=f'Task: {task}',
        config=types.GenerateContentConfig(
            system_instruction=base,
            temperature=0.0,
        )
    )
    raw = response.text.strip()

    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
    return json.loads(raw)['plan']

In [53]:
# Shared mini-executor 
def execute_subtask(subtask: dict, step_budget: int = None) -> dict:
    """Run a mini agent loop. step_budget=None means unlimited (failure mode demo)."""
    specs  = build_tool_spec(subtask.get('tools') or list(TOOLS.keys()))
    sysins = 'Research agent. Complete the subtask goal using available tools. When done, write a paragraph summary.'
    cfg    = types.GenerateContentConfig(
        system_instruction=sysins,
        tools=specs,
        temperature=0.1,
    )

    messages = [
        types.Content(role="user", parts=[types.Part(text=subtask['goal'])])
    ]
    calls, findings = [], None
    max_steps = step_budget if step_budget else 20  # 20 = effectively unlimited for demo

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL,
            contents=messages,
            config=cfg,
        )
        
        # Safety check: ensure response has candidates and content
        if not response.candidates or not response.candidates[0].content:
            findings = "API returned empty response. Unable to proceed."
            break
            
        parts = response.candidates[0].content.parts
        if not parts:
            findings = "API returned empty content parts."
            break

        part = parts[0]

        if part.function_call and part.function_call.name:
            name = part.function_call.name
            args = dict(part.function_call.args)
            calls.append((step, name, args))
            fn     = TOOLS.get(name)
            result = fn(**args) if fn else f'UNKNOWN: {name}'
            messages.append(response.candidates[0].content)
            messages.append(types.Content(
                role="user",
                parts=[types.Part(function_response=types.FunctionResponse(
                    name=name,
                    response={"result": result}
                ))]
            ))
        else:
            findings = part.text.strip() if part.text else "No findings."
            break

    if findings is None:
        # Forced summary after budget hit
        messages.append(types.Content(
            role="user",
            parts=[types.Part(text="Summarise findings.")]
        ))
        resp = client.models.generate_content(
            model=MODEL,
            contents=messages,
            config=cfg,
        )
        if resp.candidates and resp.candidates[0].content and resp.candidates[0].content.parts:
            findings = resp.candidates[0].content.parts[0].text.strip()
        else:
            findings = "Unable to generate summary."

    return {'id': subtask.get('id', '?'), 'goal': subtask['goal'],
            'calls': calls, 'findings': findings}


TASK = (
    "DocuFlow AI is a UK-based document-management SaaS (£4.2M ARR) evaluating entry "
    "into the German enterprise market. Produce a structured market entry assessment "
    "covering: (1) regulatory requirements, (2) market size and growth, "
    "(3) competitive landscape, (4) estimated Year 1 investment."
)

print("Shared setup complete.")
print(f"Tools available: {list(TOOLS.keys())}")

Shared setup complete.
Tools available: ['search_regulations', 'get_market_data', 'get_competitor_info', 'get_technical_requirements', 'estimate_financial_impact', 'search_news']


---
## Failure Mode 1: Rigid Planning

**What happens:** The plan is generated once at the start and executed without
any mechanism to revise it. When mid-execution evidence contradicts a planned
subtask, the agent ignores the contradiction and continues blindly.

In real tasks, early tool results frequently change what
later steps should do. A rigid plan cannot adapt.

**Concrete scenario:** The regulatory subtask reveals that BSI C5 certification
takes 9-15 months and costs EUR 80-140K. A good plan would add a subtask to
assess whether DocuFlow can obtain ISO 27001 (a prerequisite) in time.
A rigid plan never asks this follow-up.


In [54]:
# Rigid planning demo 

print("Failure Mode 1: Rigid Planning")
print("=" * 55)

# Step 1: Generate plan
print("\nStep 1 — Decompose task (plan generated once):")
rigid_plan = decompose(TASK)
print(f"  {len(rigid_plan)} subtasks generated. Plan is now FIXED.")
for st in rigid_plan:
    print(f"  [{st['priority']}] {st['id']}: {st['goal']}")

# Step 2: Execute regulatory subtask — reveals BSI C5 requirement
print("\nStep 2 — Executing regulatory subtask:")
reg_subtask = next((s for s in rigid_plan
                    if any(k in s['goal'].lower()
                           for k in ['regulat', 'compliance', 'legal'])),
                   rigid_plan[0])
reg_result  = execute_subtask(reg_subtask, step_budget=3)
print(f"  Findings: {reg_result['findings'][:200]}...")

# Simulate what a smart plan manager would notice
bsi_mentioned = "bsi" in reg_result['findings'].lower() or "iso 27001" in reg_result['findings'].lower()
if bsi_mentioned:
    print("\n  ⚠  KEY FINDING: BSI C5 requires ISO 27001 as prerequisite.")
    print("     A replanning agent would add: 'Assess ISO 27001 readiness timeline.'")
    print("     A RIGID agent does NOT add this — it continues with the original plan.")

# Step 3: Rigid execution continues unmodified — the follow-up is never asked
print("\nStep 3 — Rigid agent continues without revision:")
print("  Original plan unchanged. Executing remaining subtasks as-is.")
print("  No subtask added for: ISO 27001 readiness assessment")
print("  No subtask added for: BSI C5 audit timeline impact on go-to-market")
print("  No subtask added for: Phased entry strategy (sell to non-regulated sector first)")
print("\n  Result: Final report recommends immediate full market entry")
print("  without flagging the 9-15 month BSI C5 blocking dependency.")
print("  This is the RIGID PLANNING failure mode.")


Failure Mode 1: Rigid Planning

Step 1 — Decompose task (plan generated once):
  5 subtasks generated. Plan is now FIXED.
  [1] s1: Identify key regulatory requirements for SaaS and document management solutions in Germany, focusing on data privacy (GDPR), industry-specific regulations, and any relevant German laws.
  [1] s2: Determine the size and projected growth rate of the German enterprise document management and workflow automation market.
  [1] s3: Identify key competitors in the German enterprise document management SaaS market, including their market share, product offerings, and pricing strategies.
  [1] s4: Estimate the Year 1 investment required for DocuFlow AI to enter the German market, considering sales and marketing, localization, legal/compliance, and operational costs.
  [2] s5: Gather recent news and trends related to the German enterprise software market, document management, and AI adoption to inform the market entry strategy.

Step 2 — Executing regulatory subtask

### Mitigation: Replanning Triggers

After each subtask, the plan manager checks for **trigger conditions**, findings that should cause the plan to be revised.

Triggers can be keyword-based (simple) or LLM-based (powerful but slower).
Below is a keyword-trigger implementation that detects blocking dependencies.


In [55]:
# Mitigation: Replanning triggers 

REPLAN_TRIGGERS = [
    ("bsi c5",       "BSI C5 certification flagged — add: assess ISO 27001 readiness"),
    ("iso 27001",    "ISO 27001 prerequisite flagged — add: timeline assessment subtask"),
    ("9-15 months",  "Long timeline flagged — add: phased entry strategy assessment"),
    ("eur 120",      "High compliance cost flagged — add: revised Year 1 budget subtask"),
]

def check_replan_triggers(findings: str, current_plan: list) -> list:
    """Return list of new subtasks to add based on trigger conditions."""
    new_subtasks = []
    existing_goals = [s['goal'].lower() for s in current_plan]

    # Proactively augment findings with cloud security data when regulatory context detected.
    # Executors run with a tight step budget and may surface GDPR but miss BSI C5,
    # which only comes from search_regulations(germany, cloud security).
    if any(k in findings.lower() for k in ["gdpr", "bdsg", "compliance", "regulat"]):
        cloud_sec = search_regulations("germany", "cloud security")
        if "bsi c5" in cloud_sec.lower():
            findings = findings + " " + cloud_sec

    for keyword, reason in REPLAN_TRIGGERS:
        if keyword.lower() in findings.lower():
            # Avoid adding duplicate subtask
            if not any(keyword.split()[0] in g for g in existing_goals):
                print(f"  > Trigger fired: '{keyword}' → {reason}")
                # Ask the decomposer for a recovery subtask
                recovery = decompose(
                    f'Additional research needed: {reason}. Context: {findings[:300]}',
                    max_subtasks=1
                )
                new_subtasks.extend(recovery)
    return new_subtasks


# Run with replanning triggers
print("Mitigation: Replanning Triggers")
print("=" * 55)

adaptive_plan = decompose(TASK)
print(f"Initial plan: {len(adaptive_plan)} subtasks")

completed_results = []
idx = 0
while idx < len(adaptive_plan):
    st     = adaptive_plan[idx]
    print(f"\n  Executing [{st['id']}]: {st['goal'][:60]}...")
    result = execute_subtask(st, step_budget=3)
    completed_results.append(result)

    # Check for replan triggers after each subtask
    new_sts = check_replan_triggers(result['findings'], adaptive_plan)
    if new_sts:
        print(f"  + Adding {len(new_sts)} new subtask(s) to plan")
        adaptive_plan.extend(new_sts)

    idx += 1

print(f"\n  Final plan size: {len(adaptive_plan)} subtasks (started with {len(rigid_plan)})")
print(f"  Subtasks added by replanning: {len(adaptive_plan) - len(rigid_plan)}")
# Replanning triggers caught the BSI C5 blocking dependency


Mitigation: Replanning Triggers
Initial plan: 5 subtasks

  Executing [s1]: Identify key regulatory requirements for SaaS and document m...

  Executing [s2]: Determine the size and projected growth rate of the German e...

  Executing [s3]: Identify key competitors in the German enterprise document m...

  Executing [s4]: Estimate the Year 1 investment required for DocuFlow AI to e...
  > Trigger fired: 'bsi c5' → BSI C5 certification flagged — add: assess ISO 27001 readiness
  > Trigger fired: 'iso 27001' → ISO 27001 prerequisite flagged — add: timeline assessment subtask
  > Trigger fired: '9-15 months' → Long timeline flagged — add: phased entry strategy assessment
  + Adding 3 new subtask(s) to plan

  Executing [s5]: Gather recent news and trends related to the German enterpri...

  Executing [s1]: Assess DocuFlow AI's readiness for ISO 27001 certification, ...

  Executing [s1]: Assess the timeline required for DocuFlow AI to achieve ISO ...
  > Trigger fired: '9-15 months' → Lo

---
## Failure Mode 2: Over-Decomposition

**What happens:** The decomposer produces far too many subtasks. With 10-12
micro-subtasks, each with its own LLM call, the agent exhausts its context
window and token budget before reaching synthesis.

The decomposer prompt says to be thorough. Without a
subtask count limit, the LLM interprets thoroughness as more subtasks.

**Concrete symptom:** 12 subtasks, each asking for one narrow data point.
Subtasks 9-12 never execute. The final report is missing three workstreams.


In [56]:
# Over-decomposition demo 

OVER_DECOMPOSE_PROMPT = (
    'Be EXTREMELY thorough. Break the task into as many specific subtasks as possible. '
    'Each subtask should cover one narrow aspect. More subtasks = better coverage.'
)

print("Failure Mode 2: Over-Decomposition")
print("=" * 55)
print("\nDecomposing with 'be thorough' prompt (no subtask limit)...")

over_plan = decompose(TASK, instruction=OVER_DECOMPOSE_PROMPT)

print(f"\n  Decomposer produced: {len(over_plan)} subtasks")
print("\n  Full plan:")
for st in over_plan:
    print(f"    [{st['priority']}] {st['id']}: {st['goal'][:70]}")

# Simulate token budget exhaustion
SUBTASK_TOKEN_ESTIMATE = 2500  # avg tokens per subtask (LLM call + tool results)
CONTEXT_BUDGET = 15000         # conservative context budget for synthesis
max_executable = CONTEXT_BUDGET // SUBTASK_TOKEN_ESTIMATE

print(f"\n  Token budget analysis:")
print(f"    Est. tokens per subtask:   {SUBTASK_TOKEN_ESTIMATE}")
print(f"    Context budget for subts:  {CONTEXT_BUDGET}")
print(f"    Max executable subtasks:   {max_executable}")
print(f"    Planned subtasks:          {len(over_plan)}")

if len(over_plan) > max_executable:
    dropped = over_plan[max_executable:]
    print(f"\n  ✗ {len(dropped)} subtasks will NOT execute (token budget exceeded):")
    for st in dropped:
        print(f"      ✗ {st['id']}: {st['goal'][:60]}")
    print()
    print("  This is the OVER-DECOMPOSITION failure mode.")
    print("  Final report will be incomplete despite a 'thorough' plan.")

Failure Mode 2: Over-Decomposition

Decomposing with 'be thorough' prompt (no subtask limit)...

  Decomposer produced: 19 subtasks

  Full plan:
    [1] s1: Identify key UK-to-Germany regulatory differences for SaaS, focusing o
    [1] s2: Research specific German data protection laws (e.g., GDPR implementati
    [1] s3: Investigate German cybersecurity standards and certifications relevant
    [1] s4: Determine the process and requirements for a UK company to establish a
    [1] s5: Estimate the total addressable market (TAM) for document management Sa
    [1] s6: Estimate the serviceable available market (SAM) for document managemen
    [1] s7: Forecast the projected annual growth rate of the German enterprise doc
    [1] s8: Identify major competitors in the German enterprise document managemen
    [1] s9: Analyze the market share and key offerings of the top 3-5 competitors 
    [1] s10: Assess the pricing strategies and typical contract values of competito
    [1] s11: Identify a

### Mitigation: Subtask Count Cap

Enforce a maximum number of subtasks at the decomposer level.
The decomposer must consolidate related concerns into fewer, broader subtasks
that cover a full workstream.

A cap of 4-5 subtasks for a four-workstream task is a good rule of thumb.
More subtasks are only appropriate when workstreams are truly independent
and the context budget allows.


In [57]:
# Mitigation: Subtask count cap

print("Mitigation: Subtask Count Cap")
print("=" * 55)

MAX_SUBTASKS = 4

print(f"\nDecomposing with max_subtasks={MAX_SUBTASKS} cap...")
capped_plan = decompose(TASK, max_subtasks=MAX_SUBTASKS)

print(f"\n  Decomposer produced: {len(capped_plan)} subtasks")
print("  Plan:")
for st in capped_plan:
    print(f"    [{st['priority']}] {st['id']}: {st['goal'][:70]}")
    print(f"      Tools: {st['tools']}")

print(f"\n  Token budget check:")
print(f"    Subtasks: {len(capped_plan)} ≤ cap of {MAX_SUBTASKS} ✓")
print(f"    Estimated tokens: {len(capped_plan) * SUBTASK_TOKEN_ESTIMATE} (budget: {CONTEXT_BUDGET}) ✓")
# All subtasks will execute. Context budget preserved for synthesis

Mitigation: Subtask Count Cap

Decomposing with max_subtasks=4 cap...

  Decomposer produced: 4 subtasks
  Plan:
    [1] s1: Identify key regulatory requirements for SaaS and data handling in Ger
      Tools: ['search_regulations']
    [1] s2: Estimate the market size and projected growth rate for the document ma
      Tools: ['get_market_data']
    [1] s3: Analyze the competitive landscape in the German enterprise document ma
      Tools: ['get_competitor_info', 'search_news']
    [1] s4: Estimate the Year 1 investment required for DocuFlow AI to enter the G
      Tools: ['estimate_financial_impact']

  Token budget check:
    Subtasks: 4 ≤ cap of 4 ✓
    Estimated tokens: 10000 (budget: 15000) ✓


---
## Failure Mode 3: No Step Budget

**What happens:** A single subtask has no limit on how many tool calls it
can make. One ambitious subtask calls 6-8 tools, consuming most of the
available context. Later subtasks receive truncated histories or fail entirely.

Each executor is given all tools in its category and no
explicit stop condition. The model explores eagerly rather than concisely.

**Concrete symptom:** A regulatory subtask calls `search_regulations` four
times (once each for data protection, cloud security, e-invoicing, and
electronic signatures) plus `search_news` twice. 6 steps × ~800 tokens each
= 4,800 tokens for one subtask. The financial subtask that follows is
squeezed into a 600-token window and produces a one-sentence answer.


In [58]:
# No step budget demo

print("Failure Mode 3: No Step Budget")
print("=" * 55)

# An ambitious subtask with broad goal and all tools available
ambitious_subtask = {
    'id': 'regulatory_full',
    'goal': (
        'Research all regulatory and compliance requirements for DocuFlow AI '
        'operating in Germany, covering data protection, cloud security, '
        'e-invoicing, electronic signatures, and any recent regulatory updates.'
    ),
    'tools': list(TOOLS.keys()),  # ALL tools
    'priority': 1
}

print("\nRunning ambitious subtask with NO step budget...")
print(f"  Goal: {ambitious_subtask['goal'][:80]}...")
print(f"  Tools available: {ambitious_subtask['tools']}")
print()

# Run without a step budget (uses max_steps=20 internally)
no_budget_result = execute_subtask(ambitious_subtask, step_budget=None)

steps_taken = len(no_budget_result['calls'])
tokens_est  = steps_taken * 800  # rough estimate: 800 tokens per tool round-trip

print(f"\n  Steps taken (no budget):    {steps_taken}")
print(f"  Estimated tokens consumed:  ~{tokens_est:,}")
print(f"  Tool calls made:")
for step, name, args in no_budget_result['calls']:
    print(f"    Step {step}: {name}({list(args.values())[0][:40]})")

print()
print(f"  Tokens remaining for 3 more subtasks: ~{max(0, 20000 - 2000 - tokens_est):,}")
if steps_taken >= 4:
    print("  ✗ Token budget depleted. Remaining subtasks will be starved.")
    print("  This is the NO STEP BUDGET failure mode.")

Failure Mode 3: No Step Budget

Running ambitious subtask with NO step budget...
  Goal: Research all regulatory and compliance requirements for DocuFlow AI operating in...
  Tools available: ['search_regulations', 'get_market_data', 'get_competitor_info', 'get_technical_requirements', 'estimate_financial_impact', 'search_news']


  Steps taken (no budget):    1
  Estimated tokens consumed:  ~800
  Tool calls made:
    Step 1: search_regulations(Germany)

  Tokens remaining for 3 more subtasks: ~17,200


### Mitigation: Per-Subtask Step Budget

Enforce a maximum step count per subtask executor. Three is usually sufficient:
- Step 1: Core tool for this workstream
- Step 2: One follow-up or corroborating call
- Step 3: Final clarification if needed (optional)

If the step budget is hit before findings are ready, force a summary.
A constrained answer is better than starving all subsequent subtasks.


In [59]:
# Mitigation: Per-subtask step budget

print("Mitigation: Per-Subtask Step Budget (max 3 steps)")
print("=" * 55)

STEP_BUDGET = 3

print(f"\nRunning same ambitious subtask with step_budget={STEP_BUDGET}...")
budgeted_result = execute_subtask(ambitious_subtask, step_budget=STEP_BUDGET)

steps_taken_b = len(budgeted_result['calls'])
tokens_est_b  = steps_taken_b * 800

print(f"\n  Steps taken (with budget):  {steps_taken_b} ≤ {STEP_BUDGET}")
print(f"  Estimated tokens consumed:  ~{tokens_est_b:,}")
print(f"  Tokens remaining for 3 more subtasks: ~{max(0, 20000 - 2000 - tokens_est_b):,}")

print("\n  Comparison:")
print(f"    No budget:    {len(no_budget_result['calls'])} steps, ~{len(no_budget_result['calls'])*800:,} tokens")
print(f"    With budget:  {steps_taken_b} steps, ~{tokens_est_b:,} tokens")
print(f"    Tokens saved: ~{(len(no_budget_result['calls']) - steps_taken_b) * 800:,}")
print(f"    Step budget enforced. {STEP_BUDGET} subtasks × {STEP_BUDGET} steps = predictable cost.")

Mitigation: Per-Subtask Step Budget (max 3 steps)

Running same ambitious subtask with step_budget=3...

  Steps taken (with budget):  1 ≤ 3
  Estimated tokens consumed:  ~800
  Tokens remaining for 3 more subtasks: ~17,200

  Comparison:
    No budget:    1 steps, ~800 tokens
    With budget:  1 steps, ~800 tokens
    Tokens saved: ~0
    Step budget enforced. 3 subtasks × 3 steps = predictable cost.


A production planning agent should enforce all three guards simultaneously:

| Guard | Implementation | Prevents |
|---|---|---|
| Replanning triggers | Keyword scan after each subtask | Rigid planning |
| Subtask count cap | `max_subtasks=4` in decomposer prompt | Over-decomposition |
| Step budget | `step_budget=3` in executor | Runaway subtasks |

All three are **structural controls**. They cannot be replaced by prompt
instructions like "be concise" or "don't use too many tools".
The LLM will violate these instructions under pressure from the task
description asking for thoroughness.


In [60]:
# Combined: all three guards active 

print("Combined mitigation: all three guards")
print("=" * 55)

# Guard 1 + 2: capped plan
guarded_plan = decompose(TASK, max_subtasks=4)
print(f"\nDecomposed: {len(guarded_plan)} subtasks (cap=4) ✓")

guarded_completed = []
for st in sorted(guarded_plan, key=lambda x: x.get('priority', 99)):
    print(f"\n  [{st['id']}] {st['goal'][:60]}...")
    # Guard 3: step budget per subtask
    res = execute_subtask(st, step_budget=3)
    guarded_completed.append(res)

    # Guard 1: replanning triggers
    triggers = check_replan_triggers(res['findings'], guarded_plan)
    if triggers:
        print(f"    > Replanning: adding {len(triggers)} subtask(s)")
        guarded_plan.extend(triggers)

print("\n" + "=" * 55)
print(f"Completed: {len(guarded_completed)} subtasks")
total_calls = sum(len(r['calls']) for r in guarded_completed)
print(f"Total tool calls: {total_calls}")
print(f"Estimated total tokens: ~{total_calls * 800:,}")
print("\nAll three guards active: rigid planning ✓, over-decomp ✓, step budget ✓")

Combined mitigation: all three guards

Decomposed: 4 subtasks (cap=4) ✓

  [s1] Identify key regulatory requirements for SaaS and data handl...

  [s2] Estimate the market size and projected growth rate for the d...

  [s3] Analyze the competitive landscape in the German enterprise d...

  [s4] Estimate the Year 1 investment required for DocuFlow AI to e...
  > Trigger fired: 'bsi c5' → BSI C5 certification flagged — add: assess ISO 27001 readiness
  > Trigger fired: 'iso 27001' → ISO 27001 prerequisite flagged — add: timeline assessment subtask
  > Trigger fired: '9-15 months' → Long timeline flagged — add: phased entry strategy assessment
    > Replanning: adding 3 subtask(s)

Completed: 4 subtasks
Total tool calls: 4
Estimated total tokens: ~3,200

All three guards active: rigid planning ✓, over-decomp ✓, step budget ✓


## Summary

1. **Planning agents have their own failure modes.** The three shown here
   (rigid, over-decomposed, unbounded) are structural problems caused by
   missing engineering controls, not by bad prompting.

2. **Prompt instructions do not fix structural failures.** Telling the LLM
   "be flexible", "don't over-decompose", or "be concise" fails under task
   pressure. You need structural enforcement: triggers, caps, budgets.

3. **Replanning triggers must be explicit.** The system — not the LLM — must
   detect when a new finding should trigger a plan revision. Keyword scanning
   is simple and reliable; LLM-based trigger detection is more powerful
   but adds latency and cost.

4. **Step budgets scale the system.** A fleet of 4 subtasks × 3 steps each
   = 12 LLM calls, predictable cost. Without budgets, one subtask can consume
   the budget of the entire fleet.

5. **Subtask count caps force consolidation.** The decomposer must reason
   about which concerns belong together when it cannot create unlimited
   micro-subtasks. This produces more coherent, executable workstreams.